## Tests para probar la funcionalidad de import from profile

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic" 

In [2]:
import pandas as pd

from pylondrina.importing import ImportOptions
from pylondrina.schema import FieldSpec, TripSchema
from pylondrina.sources.profile import SourceProfile
from pylondrina.sources.helpers import import_trips_from_profile

### Caso A: perfil directo, sin preprocess, con correspondencia simple

In [5]:
schema = TripSchema(
    version="test-0.1",
    fields={
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=True),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=True),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
    },
    required=[
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

df = pd.DataFrame({
    "x_o": [-70.65, -70.66],
    "y_o": [-33.45, -33.44],
    "x_d": [-70.61, -70.62],
    "y_d": [-33.41, -33.42],
    "id_viaje": ["v1", "v2"],
})

profile = SourceProfile(
    name="TEST_SIMPLE",
    description="Perfil simple de prueba",
    default_field_correspondence={
        "origin_longitude": "x_o",
        "origin_latitude": "y_o",
        "destination_longitude": "x_d",
        "destination_latitude": "y_d",
        "trip_id": "id_viaje",
    },
    default_options=ImportOptions(
        keep_extra_fields=True,
        strict=False,
        strict_domains=False,
        single_stage=False,
    ),
)

dataset, report = import_trips_from_profile(
    profile=profile,
    df=df,
    schema=schema,
)

display(dataset.data.head())
print(report.ok)
print(report.field_correspondence)

,movement_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,trip_id,origin_h3_index,destination_h3_index
0,m0,-70.65,-33.45,-70.61,-33.41,v1,88b2c554e9fffff,88b2c55689fffff
1,m1,-70.66,-33.44,-70.62,-33.42,v2,88b2c55411fffff,88b2c55685fffff


True
{'origin_longitude': 'x_o', 'origin_latitude': 'y_o', 'destination_longitude': 'x_d', 'destination_latitude': 'y_d', 'trip_id': 'id_viaje'}


#### Caso B: perfil con preprocess closure

In [7]:
schema = TripSchema(
    version="test-0.1",
    fields={
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=True),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=True),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
    },
    required=[
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

df_raw = pd.DataFrame({
    "ox": [-70.65],
    "oy": [-33.45],
    "dx": [-70.61],
    "dy": [-33.41],
    "viaje": ["v100"],
})

def preprocess(df):
    out = df.copy()
    out["trip_id"] = out["viaje"]
    out["movement_id"] = out["viaje"]
    out["movement_seq"] = 0
    print("Estoy en preprocess")
    return out

profile = SourceProfile(
    name="TEST_WITH_PREPROCESS",
    description="Perfil con preprocess de prueba",
    default_field_correspondence={
        "origin_longitude": "ox",
        "origin_latitude": "oy",
        "destination_longitude": "dx",
        "destination_latitude": "dy",
        "trip_id": "trip_id",
        "movement_id": "movement_id",
        "movement_seq": "movement_seq",
    },
    preprocess=preprocess,
)

dataset, report = import_trips_from_profile(
    profile=profile,
    df=df_raw,
    schema=schema,
)

display(dataset.data)
print(report.ok)
print(dataset.provenance)

Estoy en preprocess


,origin_longitude,origin_latitude,destination_longitude,destination_latitude,viaje,trip_id,movement_id,movement_seq,origin_h3_index,destination_h3_index
0,-70.65,-33.45,-70.61,-33.41,v100,v100,v100,0,88b2c554e9fffff,88b2c55689fffff


True
{'source_profile': {'name': 'TEST_WITH_PREPROCESS', 'description': 'Perfil con preprocess de prueba'}}
